<a href="https://colab.research.google.com/github/carlosgarcia81f-create/CadenasBiblicasTematicas/blob/main/CadenasTematicasBiblicas.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd

# 1. Cargar el archivo de Excel (.xlsm)
archivo_excel = "BD_Temas.xlsm"

try:
    df = pd.read_excel(archivo_excel, sheet_name= "BD", engine='openpyxl')
    print(f"✅ Archivo cargado con éxito. Se encontraron {len(df)} registros.")
except Exception as e:
    print(f"❌ Error al cargar el archivo. Detalle: {e}")

# 2. Función con lógica de agrupación consolidada por Subtema
def mostrar_estudio_por_tema(df_base, columna_busqueda, valor_busqueda):
    """
    Filtra y muestra los versículos. En cada subtema, agrupa primero los versos
    con ideas específicas y al final recolecta todos los versos sin idea en un solo bloque.
    """
    df_base[columna_busqueda] = df_base[columna_busqueda].astype(str).str.strip()
    valor_busqueda = str(valor_busqueda).strip()

    resultado = df_base[df_base[columna_busqueda].str.contains(valor_busqueda, case=False, na=False)]

    if resultado.empty:
        print(f"⚠️ No se encontraron registros para: '{valor_busqueda}'")
        return

    # Ordenamos inicialmente para mantener consistencia
    resultado = resultado.sort_values(by=['ID_REF'])

    tema_nombre = resultado.iloc[0]['Tema_Principal']
    num_thompson = resultado.iloc[0].get('Tema Thompson', 'N/A')

    print("=" * 80)
    print(f"   ESTUDIO BÍBLICO: {tema_nombre.upper()}")
    print(f"   NÚMERO THOMPSON: {num_thompson}")
    print("=" * 80 + "\n")

    # Agrupamos por Subtema para procesar los bloques
    for subtema, grupo_subtema in resultado.groupby('Subtema', sort=False):
        subtema_str = str(subtema).strip() if pd.notna(subtema) else "SIN SUBTEMA"

        print(f"\n🔹 SUBTEMA: {subtema_str.upper()}")
        print("-" * 60)

        # Separamos: los que tienen idea de los que no
        con_idea = grupo_subtema[grupo_subtema['Ideas'].notna() & (grupo_subtema['Ideas'].astype(str).str.strip() != "")]
        sin_idea = grupo_subtema[grupo_subtema['Ideas'].isna() | (grupo_subtema['Ideas'].astype(str).str.strip() == "")]

        # 1. Mostrar versículos con Ideas agrupadas
        idea_actual = None
        for _, fila in con_idea.iterrows():
            idea_fila = str(fila['Ideas']).strip()
            if idea_fila != idea_actual:
                idea_actual = idea_fila
                print(f"\n   🔸 {idea_actual}")

            cita = f"{fila['Libro']} {fila['Capítulo']}:{fila['Versículo']}"
            texto_verso = str(fila['Texto_Verso']).replace('_x000D_', '').strip()
            print(f"         📖 {cita} -> {texto_verso}\n")

        # 2. Mostrar versículos sin idea
        if not sin_idea.empty:
            # SOLO mostrar la leyenda si existen versos con idea previa en este mismo subtema
            if not con_idea.empty:
                print(f"\n   [Versículos adicionales del subtema:]")

            for _, fila in sin_idea.iterrows():
                cita = f"{fila['Libro']} {fila['Capítulo']}:{fila['Versículo']}"
                texto_verso = str(fila['Texto_Verso']).replace('_x000D_', '').strip()
                print(f"      📖 {cita} -> {texto_verso}\n")

    print("=" * 80)
    print(f"   Fin del estudio. Total de versículos: {len(resultado)}")
    print("=" * 80)

# Ejecución
mostrar_estudio_por_tema(df, columna_busqueda='Tema_Principal', valor_busqueda='SATANÁS')


✅ Archivo cargado con éxito. Se encontraron 214 registros.
   ESTUDIO BÍBLICO: SATANÁS-ESPÍRITUS INMUNDOS
   NÚMERO THOMPSON: nan


🔹 SUBTEMA: LAS ARTIMAÑAS DE SATANÁS
------------------------------------------------------------
      📖 Marcos 4:15 -> 15 Y estos son los de junto al camino: en quienes se siembra la palabra, pero después que la oyen, en seguida viene Satanás, y quita la palabra que se sembró en sus corazones.

      📖 Marcos 8:33 -> 33 Pero él, volviéndose y mirando a los discípulos, reprendió a Pedro, diciendo: ¡Quítate de delante de mí, Satanás! porque no pones la mira en las cosas de Dios, sino en las de los hombres.

      📖 Lucas 4:6 -> 6 Y le dijo el diablo: A ti te daré toda esta potestad, y la gloria de ellos; porque a mí me ha sido entregada, y a quien quiero la doy.

      📖 1 Corintios 7:5 -> 5 No os neguéis el uno al otro, a no ser por algún tiempo de mutuo consentimiento, para ocuparos sosegadamente en la oración; y volved a juntaros en uno, para que no os t

/usr/local/lib/python3.12/dist-packages/openpyxl/worksheet/_reader.py:329: UserWarning: Data Validation extension is not supported and will be removed
  warn(msg)
